In [ ]:
pip install pennylane --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 878.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 62.5 MB/s eta 0:00:00
  Attempting uninstall: autograd
    Found existing installation: autograd 1.9.1
    Uninstalling autograd-1.9.1:
      Successfully uninstalled autograd-1.9.1


In [ ]:
import json
import pennylane as qml
import pennylane.numpy as np

def W(alpha, beta):
    norm = np.sqrt(alpha + beta)
    a = np.sqrt(alpha) / norm
    b = np.sqrt(beta) / norm
    return np.array([[a, b],
                     [b, -a]])

def controlled_U_on_0(U):
    CU = np.zeros((4,4))
    CU[0:2,0:2] = U
    CU[2,2] = 1
    CU[3,3] = 1
    return CU

def controlled_V_on_1(V):
    # Build 4x4 matrix: when aux=1 -> apply V on target
    CV = np.zeros((4,4))
    # For aux=0
    CV[0,0] = 1
    CV[1,1] = 1
    # For aux=1
    CV[2:4,2:4] = V
    return CV

dev = qml.device('default.qubit', wires=2)

@qml.qnode(dev)
def linear_combination(U, V, alpha, beta):
    qml.QubitUnitary(W(alpha, beta), wires=0)
    qml.QubitUnitary(controlled_U_on_0(U), wires=[0,1])
    qml.QubitUnitary(controlled_V_on_1(V), wires=[0,1])
    qml.QubitUnitary(qml.math.conj(qml.math.transpose(W(alpha, beta))), wires=0)
    return qml.probs(wires=[0])

# Test
test_input = '[[[0.70710678, 0.70710678],[0.70710678,-0.70710678]], [[1,0],[0,-1]], 1, 3]'
expected_output = 0.8901650422902458
out = linear_combination(*json.loads(test_input))[0]
print("Output =", out)
print("Matches expected?", np.allclose(out, expected_output, atol=1e-3))

Output = 0.8901650422902458
Matches expected? True


In [ ]:
import pennylane as qml
from pennylane import numpy as np
from scipy.linalg import expm

# Heisenberg Hamiltonian
J = 1.0

H = (
    J * (qml.PauliX(0) @ qml.PauliX(1))
    + J * (qml.PauliY(0) @ qml.PauliY(1))
    + J * (qml.PauliZ(0) @ qml.PauliZ(1))
)

H_matrix = qml.matrix(H)

eigvals, eigvecs = np.linalg.eigh(H_matrix)

ground_state = eigvecs[:, 0]

print("Ground state energy:", eigvals[0])
print("Ground state:")
print(ground_state)

Ground state energy: -3.0
Ground state:
[ 0.        +0.j  0.70710678+0.j -0.70710678+0.j  0.        +0.j]


In [ ]:
from scipy.linalg import expm

t = np.pi / 4

U = expm(-1j * H_matrix * t)

print("Shape:", U.shape)
print(U)

Shape: (4, 4)
[[7.07106781e-01-7.07106781e-01j 0.00000000e+00+0.00000000e+00j
  0.00000000e+00+0.00000000e+00j 0.00000000e+00+0.00000000e+00j]
 [0.00000000e+00+0.00000000e+00j 2.22044605e-16+0.00000000e+00j
  7.07106781e-01-7.07106781e-01j 0.00000000e+00+0.00000000e+00j]
 [0.00000000e+00+0.00000000e+00j 7.07106781e-01-7.07106781e-01j
  1.11022302e-16+1.11022302e-16j 0.00000000e+00+0.00000000e+00j]
 [0.00000000e+00+0.00000000e+00j 0.00000000e+00+0.00000000e+00j
  0.00000000e+00+0.00000000e+00j 7.07106781e-01-7.07106781e-01j]]


In [ ]:
dev = qml.device("default.qubit", wires=3)
@qml.qnode(dev)
def prepare_state():

    qml.StatePrep(ground_state, wires=[1,2])

    return qml.state()

state = prepare_state()

print(state)

[ 0.        +0.j  0.70710678+0.j -0.70710678+0.j  0.        +0.j
  0.        +0.j  0.        +0.j  0.        +0.j  0.        +0.j]


In [ ]:
dev = qml.device("default.qubit", wires=3)

@qml.qnode(dev)
def test_controlled_U():

    # Ground state
    qml.StatePrep(ground_state, wires=[1,2])

    # Ancilla qubit
    qml.Hadamard(0)

    # Controlled evolution
    qml.ctrl(
        qml.QubitUnitary(U, wires=[1,2]),
        control=0
    )

    return qml.state()

state = test_controlled_U()

print(state)

print(qml.draw(test_controlled_U)())

[ 0.        +0.j          0.5       +0.j         -0.5       +0.j
  0.        +0.j          0.        +0.j         -0.35355339+0.35355339j
  0.35355339-0.35355339j  0.        +0.j        ]
0: ──H───╭●─────┤  State
1: ─╭|Ψ⟩─├U(M0)─┤  State
2: ─╰|Ψ⟩─╰U(M0)─┤  State

M0 = 
[[7.07106781e-01-7.07106781e-01j 0.00000000e+00+0.00000000e+00j
  0.00000000e+00+0.00000000e+00j 0.00000000e+00+0.00000000e+00j]
 [0.00000000e+00+0.00000000e+00j 2.22044605e-16+0.00000000e+00j
  7.07106781e-01-7.07106781e-01j 0.00000000e+00+0.00000000e+00j]
 [0.00000000e+00+0.00000000e+00j 7.07106781e-01-7.07106781e-01j
  1.11022302e-16+1.11022302e-16j 0.00000000e+00+0.00000000e+00j]
 [0.00000000e+00+0.00000000e+00j 0.00000000e+00+0.00000000e+00j
  0.00000000e+00+0.00000000e+00j 7.07106781e-01-7.07106781e-01j]]


In [ ]:
dev = qml.device("default.qubit", wires=3, shots=1)

@qml.qnode(dev)
def iqpe_iteration():

    # Prepare eigenstate

    qml.StatePrep(ground_state, wires=[1,2])

    # Ancilla

    qml.Hadamard(0)

    qml.ctrl(
        qml.QubitUnitary(U, wires=[1,2]),
        control=0
    )

    # Final

    qml.Hadamard(0)

    return qml.sample(wires=0)

print(iqpe_iteration())

[[1]]


/usr/local/lib/python3.12/dist-packages/pennylane/devices/device_api.py:207: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


In [ ]:
from scipy.linalg import fractional_matrix_power

dev = qml.device("default.qubit", wires=3, shots=1)

@qml.qnode(dev)
def iqpe_step(U_power, feedback):

    # Prepare eigenstate
    qml.StatePrep(ground_state, wires=[1,2])

    # Ancilla
    qml.Hadamard(0)

    # Controlled U^(2^k)
    qml.ctrl(
        qml.QubitUnitary(U_power, wires=[1,2]),
        control=0
    )

    # Adaptive feedback
    qml.RZ(feedback, wires=0)

    # Interference
    qml.Hadamard(0)

    return qml.sample(wires=0)
    phase = 0

In [ ]:
bits = []

feedback = 0

for k in reversed(range(4)):

    U_power = np.linalg.matrix_power(U, 2**k)

    bit = int(iqpe_step(U_power, feedback)[0])

    bits.append(bit)

    print(f"Iteration {k}: bit = {bit}")

    feedback = -np.pi * sum(
        bits[j] / (2 ** (j + 1))
        for j in range(len(bits))
    )

print(bits)

Iteration 3: bit = 0
Iteration 2: bit = 1
Iteration 1: bit = 0
Iteration 0: bit = 0
[0, 1, 0, 0]


/usr/local/lib/python3.12/dist-packages/pennylane/ops/qubit/matrix_ops.py:300: RuntimeWarning: The two-qubit decomposition may not be differentiable when the input unitary depends on trainable parameters.
  return qp.ops.two_qubit_decomposition(U, Wires(wires))
/tmp/ipykernel_576/3469201537.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  bit = int(iqpe_step(U_power, feedback)[0])


In [ ]:
import pennylane as qml
from pennylane import numpy as np
dev = qml.device("default.qubit", wires=3)

# Heisenberg Hamiltonian

J = 1.0

H = (
    J * (qml.PauliX(1) @ qml.PauliX(2))
    + J * (qml.PauliY(1) @ qml.PauliY(2))
    + J * (qml.PauliZ(1) @ qml.PauliZ(2))
)
# Exact diagonalization

H_matrix = qml.matrix(H)
eigvals, eigvecs = np.linalg.eigh(H_matrix)
ground_state = eigvecs[:, 0]

print("Exact energy =", eigvals[0])

t = np.pi / 4

base = qml.exp(H, coeff=-1j * t)

# IQPE

shots = 1000

@qml.set_shots(shots)
@qml.qnode(dev)
def iqpe():


    # Prepare the Heisenberg ground state

    qml.StatePrep(ground_state, wires=[1,2])

    # IQPE

    bits = qml.iterative_qpe(
        base,
        aux_wire=0,
        iters=4
    )

    return qml.sample(bits)


# Execute

results = iqpe()

print(results)
majority = np.round(results.mean(axis=0)).astype(int)
print("Bits:", majority)
phase = 0

for i, b in enumerate(majority):

    phase += b / (2 ** (i + 1))

print("Estimated phase =", phase)

energy = -(2*np.pi*phase)/t

print("Recovered energy =", energy)

Exact energy = -3.0
[[0 1 1 0]
 [0 1 1 0]
 [0 1 1 0]
 ...
 [0 1 1 0]
 [0 1 1 0]
 [0 1 1 0]]
Bits: [0 1 1 0]
Estimated phase = 0.375
Recovered energy = -3.0


In [ ]:
# Iterative Quantum Phase Estimation (IQPE)
# 2-Qubit Heisenberg Hamiltonian
#
# Hamiltonian:
# H = J (X⊗X + Y⊗Y + Z⊗Z)
import pennylane as qml
from pennylane import numpy as np

dev = qml.device("default.qubit", wires=3)

J = 1.0

H = (
    J * (qml.PauliX(1) @ qml.PauliX(2))
    + J * (qml.PauliY(1) @ qml.PauliY(2))
    + J * (qml.PauliZ(1) @ qml.PauliZ(2))
)


H_matrix = qml.matrix(H)

eigvals, eigvecs = np.linalg.eigh(H_matrix)

ground_state = eigvecs[:, 0]

print("=" * 60)
print("Exact Diagonalization")
print("=" * 60)

print("\nHamiltonian Matrix:\n")
print(H_matrix)

print("\nEigenvalues:")
print(eigvals)

print("\nGround State Energy:")
print(eigvals[0])

print("\nGround State Eigenvector:")
print(ground_state)

t = np.pi / 4

base = qml.exp(H, coeff=-1j * t)

# IQPE Circuit

shots = 1000

@qml.set_shots(shots)
@qml.qnode(dev)
def iqpe():

    # Heisenberg ground state

    qml.StatePrep(ground_state, wires=[1, 2])

    # Iterative Quantum Phase Estimation

    bits = qml.iterative_qpe(
        base,
        aux_wire=0,
        iters=4
    )

    return qml.sample(bits)

# Execute IQPE

results = iqpe()

print("\n" + "=" * 60)
print("Raw IQPE Samples")
print("=" * 60)

print(results)

majority_bits = np.round(results.mean(axis=0)).astype(int)

print("\nEstimated Binary Phase:")

print(majority_bits)

phase = 0

for i, bit in enumerate(majority_bits):
    phase += bit / (2 ** (i + 1))

print("\nEstimated Phase:")

print(phase)

# Recover Energy

energy = -(2 * np.pi * phase) / t

print("\nRecovered Energy:")

print(energy)

# Error

print("\nExact Ground State Energy:")

print(eigvals[0])

print("\nAbsolute Error:")

print(abs(energy - eigvals[0]))


print("\nCircuit:\n")

print(qml.draw(iqpe, level="device")())

Exact Diagonalization

Hamiltonian Matrix:

[[ 1.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j -1.+0.j  2.+0.j  0.+0.j]
 [ 0.+0.j  2.+0.j -1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  1.+0.j]]

Eigenvalues:
[-3.  1.  1.  1.]

Ground State Energy:
-3.0

Ground State Eigenvector:
[ 0.        +0.j  0.70710678+0.j -0.70710678+0.j  0.        +0.j]

Raw IQPE Samples
[[0 1 1 0]
 [0 1 1 0]
 [0 1 1 0]
 ...
 [0 1 1 0]
 [0 1 1 0]
 [0 1 1 0]]

Estimated Binary Phase:
[0 1 1 0]

Estimated Phase:
0.375

Recovered Energy:
-3.0

Exact Ground State Energy:
-3.0

Absolute Error:
0.0

Circuit:

0: ──H───╭●─────────────────────────────────────H──┤↗│  │0⟩──H ···
1: ─╭|Ψ⟩─├(Exp(0.00-0.79j 𝓗(1.00,1.00,1.00)))⁸──────║───────── ···
2: ─╰|Ψ⟩─╰(Exp(0.00-0.79j 𝓗(1.00,1.00,1.00)))⁸──────║───────── ···
                                                    ╚═════════
                                                              
                                                              
                                   